# HKRL 远程 GPU 训练操作台

这是一份面向实际操作者的训练设备 Notebook。它负责远程 Linux/GPU 端的环境检查、配置核对、Learner/检查点服务启动、监控、恢复和评测交接。

> 默认是 `inspect` 安全模式：所有检查可运行，但不会启动常驻服务。确认参数后，把参数单元中的 `MODE` 改成 `"train"`，并把 `START_SERVICES` 改成 `True`。

## Goal

训练拓扑固定为：

```text
Windows 游戏机                         Linux / GPU 训练机
Hollow Knight + Mod                    APPO Learner : 127.0.0.1:5600
GameWorker 本地推理  ──rollout/SSH──▶  Checkpoint   : 127.0.0.1:5601
游戏环境端口 5555      ◀──weights────  runs/ssh-distributed/checkpoints
```

逐帧观察和动作永远不经过 SSH；SSH 只承载完整 rollout 和检查点。

## Setup

### 1. 一次性安装训练环境

在训练机仓库根目录运行：

```bash
bash scripts/remote/bootstrap_learner_env.sh
```

如果 CUDA 检查失败，到 <https://pytorch.org/get-started/locally/> 选择当前 Linux、Pip 和驱动支持的 CUDA wheel index，再运行：

```bash
bash scripts/remote/bootstrap_learner_env.sh \
  --torch-index-url https://download.pytorch.org/whl/<选择结果> \
  --reinstall-torch
```

不要在 GPU 训练机使用仓库根目录的 `environment.yml`，它为了 Windows/CPU worker 明确安装了 `cpuonly`。

### 2. 准备认证令牌并启动 Jupyter

令牌只保存在训练机和 Windows 会话中，不写进 Notebook：

```bash
install -d -m 700 ~/.config/hkrl
python -c 'import secrets; print(secrets.token_urlsafe(32))' > ~/.config/hkrl/auth_token
chmod 600 ~/.config/hkrl/auth_token
export HKRL_AUTH_TOKEN="$(< ~/.config/hkrl/auth_token)"
conda run --name hkrl-learner jupyter lab --no-browser
```

在 Jupyter 中选择内核 `Python (hkrl-learner)`。

## Steps

### 2. 设置本次运行参数

只需要修改这一格。首次操作建议保持单个 Gruz Mother worker，跑通后再加入其他 boss。

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shlex
import shutil
import signal
import socket
import subprocess
import sys
import time
from pathlib import Path

import torch
import yaml


def find_repo_root(start: Path) -> Path:
    resolved = start.expanduser().resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "configs").is_dir() and (candidate / "python" / "hkrl").is_dir():
            return candidate
    raise FileNotFoundError("请从 hk_Rl 仓库内启动 Jupyter，或手动设置 REPO_ROOT")


MODE = "inspect"  # inspect | train
START_SERVICES = False
STOP_SERVICES = False
RUN_REPO_CHECKS = False

REPO_ROOT = find_repo_root(Path.cwd())
TRAIN_CONFIG_REL = "configs/train/ssh_remote_learner.yaml"
TASK_FILES_REL = [
    "configs/tasks/gruz_mother.yaml",
    # 首个 boss 稳定后再取消下面两行注释：
    # "configs/tasks/hornet_protector.yaml",
    # "configs/tasks/mantis_lords.yaml",
]

RUN_DIR = REPO_ROOT / "runs" / "ssh-distributed"
STACK_LOG = RUN_DIR / "learner-stack.log"
STACK_PID = RUN_DIR / "learner-stack.pid"
AUTH_TOKEN_ENV = "HKRL_AUTH_TOKEN"

LEARNER_PORT = 5600
REGISTRY_PORT = 5601
WINDOWS_ENV_PORT = 5555
SSH_TARGET = "CHANGE_ME_user@gpu-host"

assert MODE in {"inspect", "train"}
os.chdir(REPO_ROOT)
print({"mode": MODE, "repo_root": str(REPO_ROOT), "kernel_python": sys.executable})

### 3. 检查 Python、PyTorch、驱动和 GPU

`MODE="train"` 时 CUDA 不可用会直接中止，避免误用 CPU 跑数小时。PyTorch 官方建议用 `torch.cuda.is_available()` 判断当前安装是否真正可用。

In [ ]:
system_summary = {
    "hostname": platform.node(),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
}
if torch.cuda.is_available():
    system_summary["cuda_devices"] = [
        torch.cuda.get_device_name(index)
        for index in range(torch.cuda.device_count())
    ]

print(json.dumps(system_summary, indent=2, ensure_ascii=False))

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total,memory.used,utilization.gpu",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    print("nvidia-smi:\n" + (result.stdout.strip() or result.stderr.strip()))
else:
    print("nvidia-smi: 未安装或不在 PATH")

if MODE == "train" and not torch.cuda.is_available():
    raise RuntimeError("训练模式要求 CUDA；请先修复 PyTorch/CUDA 安装")

### 4. 加载并核对合并后的配置

YAML 使用 `defaults` 深度合并。当前继承顺序是：

`base.yaml → ppo_attention_gru.yaml → remote_learner.yaml → ssh_remote_learner.yaml`

最终配置由代码打印；不要只看最外层文件。未知字段会被拒绝。

In [ ]:
from hkrl.utils.config import (
    load_task_config,
    load_train_config,
    validate_task_collection,
)

train_config_path = (REPO_ROOT / TRAIN_CONFIG_REL).resolve()
task_paths = [(REPO_ROOT / item).resolve() for item in TASK_FILES_REL]
for required_path in (train_config_path, *task_paths):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

train_cfg = load_train_config(train_config_path)
tasks = [load_task_config(path) for path in task_paths]
validate_task_collection(tasks, context="notebook tasks")

layout = {
    (
        task.observation.max_entities,
        task.observation.tier,
        task.action.enable_macro_actions,
        task.action.n_macro_actions,
    )
    for task in tasks
}
if len(layout) != 1:
    raise ValueError("所有任务必须共享 max_entities/tier/macro 动作布局")
if train_cfg.model.entity_hidden % train_cfg.model.attention_heads != 0:
    raise ValueError("model.entity_hidden 必须能被 attention_heads 整除")
if (
    train_cfg.algorithm == "appo"
    and train_cfg.learner.publish_every_updates > train_cfg.learner.max_staleness + 1
):
    raise ValueError("APPO 检查点发布过慢，会在 worker 更新权重前耗尽陈旧窗口")

config_summary = {
    "algorithm": train_cfg.algorithm,
    "device": train_cfg.learner.device,
    "model": train_cfg.model.model_dump(mode="json"),
    "rollout_steps": train_cfg.rollout_steps,
    "minibatch_size": train_cfg.minibatch_size,
    "epochs": train_cfg.epochs,
    "gamma": train_cfg.gamma,
    "gae_lambda": train_cfg.gae_lambda,
    "learning_rate": train_cfg.learning_rate,
    "clip_range": train_cfg.clip_range,
    "entropy_coef": train_cfg.entropy_coef,
    "max_staleness": train_cfg.learner.max_staleness,
    "publish_every_updates": train_cfg.learner.publish_every_updates,
    "checkpoint_dir": train_cfg.learner.checkpoint_dir,
    "tasks": [task.task_id for task in tasks],
}
print(yaml.safe_dump(config_summary, sort_keys=False, allow_unicode=True))

if MODE == "train" and not train_cfg.learner.device.startswith("cuda"):
    raise RuntimeError("训练配置必须显式使用 cuda/cuda:N")

### 核心配置说明

| 配置 | 当前值 | 含义与调节原则 |
|---|---:|---|
| `algorithm` | `appo` | 异步接收 Windows worker rollout，并按策略版本过滤陈旧数据。 |
| `gamma` | `0.995` | 长期回报折扣。Boss 战较长，因此比常见的 0.99 更重视后续结果。 |
| `gae_lambda` | `0.95` | GAE 偏差/方差折中；首轮不要改。 |
| `rollout_steps` | `2048` | 每个 worker 每次上传的 transition 数。越大网络开销越低，但权重更新更慢。 |
| `minibatch_size` | `256` | GPU 单次反向传播样本数。显存不足先降到 128；利用率过低可升到 512。 |
| `epochs` | `4` | 每批 rollout 重用次数。KL 快速上升时降到 2。 |
| `learning_rate` | `3e-4` | Adam 学习率。策略抖动或 KL 过高时优先降到 `1e-4`。 |
| `clip_range` | `0.2` | PPO ratio 裁剪范围。首轮保持。 |
| `entropy_coef` | `0.01` | 探索强度。动作长期随机可逐步降低；过早僵化可提高。 |
| `value_coef` | `0.5` | value loss 权重。 |
| `max_grad_norm` | `0.5` | 梯度裁剪，防止偶发爆炸。 |
| `max_staleness` | `4` | rollout 策略最多落后 learner 多少个版本。 |
| `publish_every_updates` | `4` | 每 4 次更新发布权重；必须 `<= max_staleness + 1`。 |
| `learner.device` | `cuda` | GPU 配置硬性要求 CUDA，CUDA 不可用时拒绝启动。 |
| `action_repeat` | `2` | 一个动作持续两个 FixedUpdate；影响真实控制频率和 SPS。 |
| `observation.tier` | `privileged` | 使用内部状态的主训练层级；评测时再做 reduced/human-visible 消融。 |

模型 `entity_attention_gru` 使用实体注意力 + GRU；`entity_mask` 屏蔽填充实体。当前分布式 APPO 上传的是扁平 transition 及每步 GRU state，更新是单步 recurrent-state 条件学习，并非完整序列 BPTT。`sequence_length`/`burn_in` 主要用于本地 `recurrent_ppo` 路径。

In [ ]:
samples_per_update = train_cfg.rollout_steps
minibatches_per_epoch = math.ceil(samples_per_update / train_cfg.minibatch_size)
optimizer_steps_per_update = minibatches_per_epoch * train_cfg.epochs
derived = {
    "samples_per_worker_rollout": samples_per_update,
    "minibatches_per_epoch": minibatches_per_epoch,
    "optimizer_steps_per_accepted_rollout": optimizer_steps_per_update,
    "checkpoint_after_accepted_rollouts": train_cfg.learner.publish_every_updates,
}
print(json.dumps(derived, indent=2, ensure_ascii=False))

### 5. 检查认证令牌

只打印令牌指纹，不打印令牌本身。Windows 上必须设置完全相同的 `HKRL_AUTH_TOKEN`。

In [ ]:
auth_token = os.environ.get(AUTH_TOKEN_ENV, "")
token_summary = {
    "environment_variable": AUTH_TOKEN_ENV,
    "present": bool(auth_token),
    "fingerprint": (
        hashlib.sha256(auth_token.encode("utf-8")).hexdigest()[:12]
        if auth_token
        else None
    ),
}
print(token_summary)
if MODE == "train" and not auth_token:
    raise RuntimeError(f"训练模式要求先设置 {AUTH_TOKEN_ENV}")

### 6. 可选：运行仓库门禁

首次部署或拉取新代码后，把 `RUN_REPO_CHECKS=True`，执行 schema、lint、mypy 和 pytest。该检查不会连接 Hollow Knight。

In [ ]:
if RUN_REPO_CHECKS:
    subprocess.run(["make", "check"], cwd=REPO_ROOT, check=True)
else:
    print("跳过 make check；需要时把 RUN_REPO_CHECKS 改为 True 后重跑本格。")

### 7. 启动远端 Learner + Checkpoint 服务

服务只监听 `127.0.0.1:5600/5601`。Notebook 会把 supervisor PID 和日志放到 `runs/ssh-distributed/`。重复执行时若旧进程仍存活会拒绝启动第二套。

In [ ]:
def pid_is_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
    except (OSError, ProcessLookupError):
        return False
    return True


def port_is_open(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.25):
            return True
    except OSError:
        return False


stack_command = [
    "bash",
    str(REPO_ROOT / "scripts" / "remote" / "start_learner_stack.sh"),
    *[str(path) for path in task_paths],
]
print("启动命令：", shlex.join(stack_command))

if START_SERVICES:
    if MODE != "train":
        raise RuntimeError("启动服务前必须把 MODE 改为 'train'")
    if not auth_token:
        raise RuntimeError(f"缺少 {AUTH_TOKEN_ENV}")
    if STACK_PID.exists():
        existing_pid = int(STACK_PID.read_text(encoding="utf-8").strip())
        if pid_is_alive(existing_pid):
            raise RuntimeError(f"训练栈已在运行，PID={existing_pid}")

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    child_env["HKRL_PYTHON_BIN"] = sys.executable
    child_env["HKRL_TRAIN_CONFIG"] = str(train_config_path)
    log_handle = STACK_LOG.open("ab", buffering=0)
    process = subprocess.Popen(
        stack_command,
        cwd=REPO_ROOT,
        env=child_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_handle.close()
    STACK_PID.write_text(f"{process.pid}\n", encoding="utf-8")

    deadline = time.monotonic() + 30.0
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = STACK_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-40:]
            raise RuntimeError("训练栈提前退出：\n" + "\n".join(tail))
        if port_is_open(LEARNER_PORT) and port_is_open(REGISTRY_PORT):
            break
        time.sleep(0.5)
    print({
        "pid": process.pid,
        "learner_ready": port_is_open(LEARNER_PORT),
        "registry_ready": port_is_open(REGISTRY_PORT),
        "log": str(STACK_LOG),
    })
else:
    print("安全模式：未启动服务。确认配置后设置 MODE='train'、START_SERVICES=True。")

### 8. Windows 端要执行的命令

先确保 Windows PowerShell 已设置相同令牌，然后保持 SSH 隧道窗口运行。初次验收使用 `-Steps 2048`，只上传一个完整 rollout；确认更新成功后再去掉该参数连续训练。

In [ ]:
windows_tunnel = (
    '.\\scripts\\windows\\start_ssh_tunnel.ps1 '
    f'-Remote "{SSH_TARGET}" '
    f'-LocalLearnerPort {LEARNER_PORT} -LocalRegistryPort {REGISTRY_PORT}'
)
windows_worker_once = (
    '.\\scripts\\windows\\start_game_worker.ps1 '
    '-Config "configs/train/windows_game_worker.yaml" '
    '-Task "configs/tasks/gruz_mother.yaml" '
    '-WorkerId "windows-game-0" -Steps 2048'
)
windows_worker_continuous = windows_worker_once.rsplit(" -Steps", 1)[0]

print("PowerShell 窗口 1（保持运行）：\n" + windows_tunnel)
print("\nPowerShell 窗口 2（首次仅一个 rollout）：\n" + windows_worker_once)
print("\n确认成功后连续训练：\n" + windows_worker_continuous)

## Checks

### 9. 查看训练状态

重复运行本格即可刷新。重点看：checkpoint/policy version 是否增加、`policy_kl` 是否有限、loss/gradient 是否出现 NaN、GPU 是否有利用率。Learner 日志每个 rollout 会输出一行 JSON。

In [ ]:
checkpoint_dir = (REPO_ROOT / train_cfg.learner.checkpoint_dir).resolve()
index_path = checkpoint_dir / "index.jsonl"
latest_meta = None

if STACK_PID.exists():
    try:
        stack_pid_value = int(STACK_PID.read_text(encoding="utf-8").strip())
    except ValueError:
        stack_pid_value = -1
else:
    stack_pid_value = -1

service_status = {
    "pid": stack_pid_value if stack_pid_value > 0 else None,
    "pid_alive": stack_pid_value > 0 and pid_is_alive(stack_pid_value),
    "learner_port_open": port_is_open(LEARNER_PORT),
    "registry_port_open": port_is_open(REGISTRY_PORT),
    "checkpoint_dir": str(checkpoint_dir),
}
print(json.dumps(service_status, indent=2, ensure_ascii=False))

if index_path.is_file():
    records = [
        json.loads(line)
        for line in index_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if records:
        latest_meta = max(records, key=lambda item: int(item["version"]))
        latest_path = checkpoint_dir / latest_meta["path"]
        payload = torch.load(latest_path, map_location="cpu", weights_only=True)
        checkpoint_summary = {
            "registry_version": latest_meta["version"],
            "policy_version": payload.get("policy_version"),
            "update": payload.get("update"),
            "sha256": latest_meta["sha256"],
            "optimizer_saved": "optimizer_state_dict" in payload,
            "metrics": payload.get("metrics", {}),
            "size_mib": round(latest_path.stat().st_size / 1024**2, 2),
        }
        print("latest checkpoint:\n" + json.dumps(checkpoint_summary, indent=2, ensure_ascii=False))
else:
    print("尚无 index.jsonl；Learner 首次成功启动后会发布初始化检查点。")

if STACK_LOG.is_file():
    log_lines = STACK_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\nlearner log tail:\n" + "\n".join(log_lines[-30:]))

if shutil.which("nvidia-smi"):
    gpu_now = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,memory.used,memory.total,utilization.gpu,temperature.gpu",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    print("\nGPU now:\n" + gpu_now.stdout.strip())

### 指标怎么判断

- `policy_loss` / `value_loss`：必须有限；单次绝对值不是能力结论，关注连续趋势和异常跳变。
- `policy_kl`：持续大幅上升说明一次更新改得太猛，先降低 `learning_rate` 或 `epochs`。
- `action_entropy`：快速接近零通常表示过早坍缩；长期很高且胜率不升表示策略仍近似随机。
- `explained_variance`：从接近 0 往正值提高表示 value 预测开始有用；负值表示比常数预测更差。
- `grad_norm`：应有限；频繁顶到裁剪上限需要检查奖励尺度和学习率。
- `samples`：当前每个被接受 rollout 的样本数。

这些只是训练健康指标。模型能力必须在 Windows 游戏端用固定 seed、隔离评测得到 boss 胜率、受伤量和击杀时间。

### 10. 安全停止服务

把顶部参数 `STOP_SERVICES=True` 后运行下一格。它只会终止 PID 文件对应且命令行包含 `start_learner_stack.sh` 的进程组。

In [ ]:
if STOP_SERVICES:
    if not STACK_PID.is_file():
        print("没有 PID 文件，训练栈可能已经停止。")
    else:
        pid_to_stop = int(STACK_PID.read_text(encoding="utf-8").strip())
        if not pid_is_alive(pid_to_stop):
            print(f"PID {pid_to_stop} 已不存在。")
            STACK_PID.unlink(missing_ok=True)
        else:
            cmdline_path = Path(f"/proc/{pid_to_stop}/cmdline")
            cmdline = (
                cmdline_path.read_bytes().replace(b"\x00", b" ").decode("utf-8", "replace")
                if cmdline_path.is_file()
                else ""
            )
            if "start_learner_stack.sh" not in cmdline:
                raise RuntimeError(f"拒绝终止不匹配的 PID {pid_to_stop}: {cmdline}")
            os.killpg(os.getpgid(pid_to_stop), signal.SIGTERM)
            deadline = time.monotonic() + 15.0
            while pid_is_alive(pid_to_stop) and time.monotonic() < deadline:
                time.sleep(0.25)
            STACK_PID.unlink(missing_ok=True)
            print(f"训练栈 PID {pid_to_stop} 已停止。")
else:
    print("未停止服务；需要停止时设置 STOP_SERVICES=True。")

## Next Steps

### 恢复训练

再次执行启动格即可。Learner 会验证最新 checkpoint 的 SHA-256，然后恢复：

- 模型参数；
- policy/update 版本；
- Adam optimizer state；
- 最近一次训练 metrics。

不要删除 `index.jsonl` 中仍被引用的 checkpoint。若要开启完全独立的新实验，请改用新的 `learner.checkpoint_dir`，而不是覆盖旧目录。

### 固定种子评测交接到 Windows

训练机没有游戏环境，因此不能在这里评测 boss 能力。停止训练或选定一个明确版本后，将该 checkpoint 复制到 Windows，先核对 SHA-256，再运行 `run_eval.py`。下一格会根据最新 registry 条目生成命令模板。

In [ ]:
if latest_meta is None:
    print("当前没有可交接的 checkpoint。")
else:
    remote_checkpoint = checkpoint_dir / latest_meta["path"]
    windows_checkpoint = f'.\\runs\\evaluation\\{latest_meta["path"]}'
    print("Windows PowerShell：")
    print("New-Item -ItemType Directory -Force .\\runs\\evaluation | Out-Null")
    print(f'scp "{SSH_TARGET}:{remote_checkpoint}" "{windows_checkpoint}"')
    print(f'(Get-FileHash -Algorithm SHA256 "{windows_checkpoint}").Hash.ToLower()')
    print(f'期望 SHA-256: {latest_meta["sha256"].lower()}')
    print()
    print(
        'conda run --name hkrl python scripts/run_eval.py '
        '--policy model '
        f'--checkpoint "{windows_checkpoint}" '
        '--train-config configs/train/windows_game_worker.yaml '
        '--tasks configs/tasks/gruz_mother.yaml '
        '--episodes 20 --seeds 0 1 2 '
        '--host 127.0.0.1 --port 5555 '
        '--output runs/evaluation/gruz_mother.json'
    )

### 推荐实际顺序

1. `inspect` 模式跑完所有格，确认配置和 CUDA。
2. 训练机启动服务，确认日志出现 `learner_ready` 和初始化 checkpoint。
3. Windows 建立 SSH 隧道，先运行一个 2048-step Gruz Mother rollout。
4. 训练日志应出现 `learner_update`；连续 4 次更新后应发布新 checkpoint。
5. 连续训练前先做 20 局固定 seed 基线评测。
6. Gruz Mother 的胜率、重置成功率和无效动作比稳定后，再加入 Hornet 与 Mantis Lords。
7. 每次调参只改一组变量，并使用新的 checkpoint 目录保存实验边界。